In [ ]:
#----------------------------------------------------------#
#             Program: Eartquake 2021/07/30                #
#               All rights reserved 2021                   #
#----------------------------------------------------------#
#     From: Ekobots Innovation Ltda - www.ekobots.com      #
#       by: Juan Sirgado y Antico - www.jsya.com.br        #
#----------------------------------------------------------#
# Description:                                             #
# Global map with recent earthquakes Week/Month            #
#----------------------------------------------------------#
# Site: http://jsirgado.pythonanywhere.com/                #
#----------------------------------------------------------#

SyntaxError: unexpected EOF while parsing (TEMP/ipykernel_4428/568663850.py, line 20)

In [ ]:
#----------------------------------------------------------#
# def hello_world():
#    return 'Hello from Flask!'
#----------------------------------------------------------#
from flask import Flask
app = Flask(__name__)
@app.route('/')
#----------------------------------------------------------#
def earthquakes():

    import datetime
    import urllib
    import folium
    import os
    import webbrowser

    # Arquivos de terremotos da internet
    # https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_month.csv
    # https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_week.csv
    # https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_day.csv
    # https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_hour.csv

    # Monta o arquivo de dados em uma lista
    def newList(nlist,nfile):
        for line in nfile:
            sline = str(line.decode("utf-8")).replace("\n","")
            nline = sline.split(",")
            if nline[0] != "time":
                nlist.append(nline)

    # Monta a lista de terremotos da semana com base no arquivo csv da internet
    wfile = urllib.request.urlopen("https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_week.csv")
    wlist = []
    newList(wlist,wfile)

    # Monta a lista de terremotos do mes com base no arquivo csv da internet
    mfile = urllib.request.urlopen("https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_month.csv")
    mlist = []
    newList(mlist,mfile)

    # Inverte a ordem dos terremotos os mais antigos primeiro
    wlist.reverse()
    mlist.reverse()

    # Cria o mapa base para os dados
    fmap = folium.Map(location=[0, 0],
                    zoom_start=2.5,
                    tiles="OpenStreetMap")
    
    def circlemark(data,group,type):
        # Monta/Formata os dados/estilo dos terremotos no mapa
        cDate = data[0][0:10]
        cTime = data[0][11:22]
        cLong = float(data[1].strip())
        cLati = float(data[2].strip())
        cDeep = data[3].strip()
        cMag = float(data[4].strip())
        cRadius = pow(float(cMag),2) # * 10000
        cPlace = data[13].strip()
        cPopup = "Date:" + cDate + "\nTime:" + cTime + "\nMag:" + str(cMag) + "\nDeep:" + cDeep + "Km\nPlace:" + cPlace

        # Seleciona a cor (Today=red, Yesterday=orange, before=yellow)
        dInput = datetime.date(int(cDate[0:4]), int(cDate[5:7]), int(cDate[8:10]))
        dToday = datetime.datetime.today().date()
        # dToday = datetime.datetime.utcnow().date()
        difDays = (dToday - dInput).days
        if difDays == 0:
            cColor = "red"
        elif difDays == 1:
            cColor = "orange"
        else:
            cColor = "yellow"
        if type == "M":
            cColor = "magenta"

        # Cria o circulo/ponto marcando o epicentro do terremoto
        folium.CircleMarker(
            radius=1 if type == "M" else cRadius,
            weight=3 if type == "M" else 1,
            popup=cPopup,
            location=[cLong, cLati],
            color=cColor,
            opacity=1,
            fill_color=cColor,
            fill_opacity=0.1,
            fill=True).add_to(group)
        return

    # Cria um ponto no local do terremoto
    mgroup = folium.FeatureGroup(name='Earthquakes').add_to(fmap)
    for row in mlist:
        circlemark(row,mgroup,"M")

    # Cria o circulo baseado na magnetude do terremoto
    wgroup = folium.FeatureGroup(name='Magnitude').add_to(fmap)
    for row in wlist:
        circlemark(row,wgroup,"W")

    # Monta/Formata o estilo das bordas tectonicas no mapa
    def tract_styles(feature):
        return {"fillColor": "green",
                "color": "brown",
                "weight": 1,
                "dashArray": "5, 5",
                "fillOpacity": 1}

    # Inclui as bordas tectonicas no mapa
    folium.GeoJson("https://raw.githubusercontent.com/fraxen/tectonicplates/master/GeoJSON/PB2002_boundaries.json",
                name="Tectonics",
                style_function=tract_styles).add_to(fmap)

    # Cria/Formata a imagem do USGS earthquakes
    iusgs = os.path.join("USGS_Earthquakes.png")
    ioverlay = folium.raster_layers.ImageOverlay(
        name="USGS",
        image=iusgs,
        bounds=[[-80, 200], [-70, 140]],
        opacity=0.5,
        interactive=True,
        cross_origin=False,
        zindex=1)

    # Inclui a imagem do USGS earthquakes no mapa
    folium.Popup("https://earthquake.usgs.gov").add_to(ioverlay)
    ioverlay.add_to(fmap)
    # Inclui o controle de layers no mapa
    folium.LayerControl(
            position="topright",
            collapsed=False,
            autoZIndex=True).add_to(fmap)

    # Salva o mapa com os terremotos atualizados
    musgs = os.path.join("USGS_Earthquakes.html")
    fmap.save(musgs)
    # Abre o mapa com os terremotos atualizados no browser
    webbrowser.open(musgs)

    # Monta HTML do Mapa para apresentacao
    content = fmap.get_root().render()
    return content
#----------------------------------------------------------#
emap = earthquakes()
# Abre o mapa com os terremotos no browser
#emap
#----------------------------------------------------------#

In [7]:
#----------------------------------------------------------#
# That is all folks!                                       #
#----------------------------------------------------------#